# Test Skill Extraction Model

This notebook allows you to manually input text and see what KSAOs are extracted.

**Features:**
- Manual text input
- Scans ALL KSAO categories in the knowledge base
- Shows detailed extraction results with similarity scores
- Displays skills by category

## Setup

In [ ]:
import sys
sys.path.append('..')

from skillner.jd_skill_extractor import JobDescriptionSkillExtractor
from skillner.onet_converter import load_kb
import json

print("✓ Imports successful")

## Configuration

Edit the knowledge base path if needed:

In [ ]:
# Knowledge base path
KB_PATH = '../.skillner-kb/MERGED_EN.pkl'  # Or ONET_EN.pkl

# Extraction parameters
SIMILARITY_THRESHOLD = 0.6  # Lower = more matches (e.g., 0.5), Higher = stricter (e.g., 0.7)
MAX_WINDOW_SIZE = 5  # Maximum words in a skill phrase

print(f"Configuration:")
print(f"  Knowledge base: {KB_PATH}")
print(f"  Similarity threshold: {SIMILARITY_THRESHOLD}")
print(f"  Max window size: {MAX_WINDOW_SIZE}")

## Check Knowledge Base Coverage

First, let's see what KSAO categories are in the knowledge base:

In [ ]:
# Load knowledge base
print("Loading knowledge base...")
kb = load_kb(KB_PATH)
print(f"✓ Loaded {len(kb):,} unique skill entries\n")

# Count skills by section
section_counts = {}
all_sections = set()

for skill_name, entries in kb.items():
    for entry in entries:
        section = entry.get('section', 'Unknown')
        all_sections.add(section)
        section_counts[section] = section_counts.get(section, 0) + 1

print("="*70)
print("KSAO Categories in Knowledge Base")
print("="*70)
for section in sorted(section_counts.keys()):
    count = section_counts[section]
    print(f"  [{section:30s}]: {count:6,} skills")

print(f"\nTotal sections: {len(all_sections)}")
print(f"Total skill entries: {sum(section_counts.values()):,}")
print(f"Unique skill names: {len(kb):,}")

## Initialize Extractor

This will load the semantic model (may take 1-2 minutes):

In [ ]:
print("Initializing extractor...")
print("(This may take 1-2 minutes to load the semantic model)\n")

extractor = JobDescriptionSkillExtractor(
    kb_path=KB_PATH,
    model_name='all-MiniLM-L6-v2',
    similarity_threshold=SIMILARITY_THRESHOLD,
    max_window_size=MAX_WINDOW_SIZE
)

print("✓ Extractor ready!")
print(f"  Model: all-MiniLM-L6-v2")
print(f"  Knowledge base size: {len(extractor.kb):,} skills")
print(f"  Similarity threshold: {SIMILARITY_THRESHOLD}")

## Test with Example Text

Try the pre-loaded example first:

In [ ]:
# Example text
example_text = """
Position Summary: The Food Service Worker is responsible for preparing and serving food items 
while providing excellent customer service and adhering to all food safety procedures. This role 
requires strong communication skills, attention to detail, and the ability to work in a fast-paced 
environment. The ideal candidate will have experience with food preparation, knowledge of proper 
sanitation practices, and proficiency in basic math for handling transactions. Must be able to 
operate kitchen equipment safely and work effectively as part of a team. Good time management and 
organizational skills are essential. Maintains excellent customer service and positive attitude 
towards guests and co-workers. Uses critical thinking to solve problems and make decisions quickly.
"""

print("Example Text:")
print("="*70)
print(example_text.strip())
print("="*70)

In [ ]:
# Extract skills
print("\nExtracting skills...\n")
result = extractor.extract_skills(example_text, return_details=True)

print("="*70)
print("EXTRACTION RESULTS")
print("="*70)
print(f"\nTotal unique skills found: {result['num_skills']}")

# Show by category
print(f"\n{'='*70}")
print("Skills by KSAO Category")
print("="*70)

for section, skills in sorted(result['by_section'].items()):
    if skills:
        print(f"\n[{section}]: {len(skills)} skills")
        for i, skill in enumerate(sorted(skills), 1):
            print(f"  {i:2d}. {skill}")

# Show detailed matches with similarity scores
if result.get('details'):
    print(f"\n\n{'='*70}")
    print("Detailed Matches (with similarity scores)")
    print("="*70)
    
    for detail in result['details']:
        skill_name = detail['skill']
        section = detail['section']
        matched_text = detail.get('matched_text', 'N/A')
        similarity = detail.get('similarity_score', 0)
        
        print(f"\n• Skill: {skill_name}")
        print(f"  Section: {section}")
        print(f"  Matched text: '{matched_text}'")
        print(f"  Similarity: {similarity:.3f}")

# Summary
print(f"\n\n{'='*70}")
print("Summary")
print("="*70)
print(f"All extracted skills (alphabetical):")
for i, skill in enumerate(sorted(result['skills']), 1):
    print(f"{i:2d}. {skill}")

## Manual Input Test

**Edit the text below** to test with your own job description or text:

In [ ]:
# =====================================================================
# EDIT THIS TEXT - Put your own job description or text here:
# =====================================================================

my_text = """
Put your text here. For example:

We are seeking a Software Engineer with strong programming skills in Python and Java.
The ideal candidate should have experience with database management, cloud computing,
and excellent problem-solving abilities. Must be able to work in teams, communicate
effectively with stakeholders, and demonstrate critical thinking skills. Knowledge of
machine learning and data analysis is a plus. The role requires attention to detail,
time management, and the ability to learn new technologies quickly.
"""

# =====================================================================

print("Your Text:")
print("="*70)
print(my_text.strip())
print("="*70)

In [ ]:
# Extract skills from your text
print("\nExtracting skills from your text...\n")
my_result = extractor.extract_skills(my_text, return_details=True)

print("="*70)
print("EXTRACTION RESULTS")
print("="*70)
print(f"\nTotal unique skills found: {my_result['num_skills']}")

# Show by category
print(f"\n{'='*70}")
print("Skills by KSAO Category")
print("="*70)

for section, skills in sorted(my_result['by_section'].items()):
    if skills:
        print(f"\n[{section}]: {len(skills)} skills")
        for i, skill in enumerate(sorted(skills), 1):
            print(f"  {i:2d}. {skill}")

# Show detailed matches
if my_result.get('details'):
    print(f"\n\n{'='*70}")
    print("Detailed Matches (with similarity scores)")
    print("="*70)
    
    for detail in sorted(my_result['details'], key=lambda x: x.get('similarity_score', 0), reverse=True):
        skill_name = detail['skill']
        section = detail['section']
        matched_text = detail.get('matched_text', 'N/A')
        similarity = detail.get('similarity_score', 0)
        
        print(f"\n• Skill: {skill_name}")
        print(f"  Section: {section}")
        print(f"  Matched text: '{matched_text}'")
        print(f"  Similarity: {similarity:.3f}")

# Summary
print(f"\n\n{'='*70}")
print("Summary - All Extracted Skills (alphabetical)")
print("="*70)
for i, skill in enumerate(sorted(my_result['skills']), 1):
    print(f"{i:2d}. {skill}")

## Adjust Parameters

If you want to try different similarity thresholds:

In [ ]:
# Try different thresholds
thresholds_to_test = [0.5, 0.6, 0.7, 0.8]

print("Testing different similarity thresholds on your text:")
print("="*70)

for threshold in thresholds_to_test:
    # Create new extractor with this threshold
    test_extractor = JobDescriptionSkillExtractor(
        kb_path=KB_PATH,
        model_name='all-MiniLM-L6-v2',
        similarity_threshold=threshold,
        max_window_size=MAX_WINDOW_SIZE
    )
    
    # Extract skills
    test_result = test_extractor.extract_skills(my_text, return_details=False)
    
    print(f"\nThreshold {threshold:.1f}: {test_result['num_skills']} skills found")
    
    # Show breakdown by section
    for section, skills in sorted(test_result['by_section'].items()):
        if skills:
            print(f"  [{section}]: {len(skills)}")

print("\n" + "="*70)
print("Recommendation:")
print("  - Lower threshold (0.5-0.6): More matches, may include some false positives")
print("  - Medium threshold (0.6-0.7): Balanced, recommended for most use cases")
print("  - Higher threshold (0.7-0.8): Stricter, fewer but more confident matches")
print("="*70)

## Compare Multiple Texts

Test multiple job descriptions at once:

In [ ]:
# Add multiple texts to compare
texts_to_compare = [
    {
        'name': 'Food Service Worker',
        'text': example_text
    },
    {
        'name': 'Software Engineer',
        'text': """Seeking a Software Engineer with expertise in Python, Java, and cloud computing.
        Strong problem-solving skills and experience with agile development required.
        Must have knowledge of database systems, API development, and version control."""
    },
    {
        'name': 'Marketing Manager',
        'text': """Marketing Manager needed with strong communication and leadership skills.
        Experience in digital marketing, social media strategy, and data analysis required.
        Must demonstrate creativity, strategic thinking, and project management abilities."""
    }
]

# Extract skills from all texts
print("Comparing multiple job descriptions:")
print("="*70)

comparison_results = []
for item in texts_to_compare:
    result = extractor.extract_skills(item['text'], return_details=False)
    comparison_results.append({
        'name': item['name'],
        'num_skills': result['num_skills'],
        'skills': result['skills'],
        'by_section': result['by_section']
    })

# Display comparison
for comp in comparison_results:
    print(f"\n[{comp['name']}]")
    print(f"  Total skills: {comp['num_skills']}")
    for section, skills in sorted(comp['by_section'].items()):
        if skills:
            print(f"    {section}: {len(skills)}")
    print(f"  Top 5 skills: {', '.join(sorted(comp['skills'])[:5])}")

print("\n" + "="*70)

## Verify All KSAO Categories Are Scanned

Check that the extractor is scanning all available categories:

In [ ]:
print("Verification: KSAO Categories Coverage")
print("="*70)

# Get all sections from knowledge base
kb_sections = set()
for entries in extractor.kb.values():
    for entry in entries:
        kb_sections.add(entry.get('section', 'Unknown'))

print(f"\nKnowledge base contains {len(kb_sections)} sections:")
for section in sorted(kb_sections):
    print(f"  ✓ {section}")

print(f"\n" + "="*70)
print("Confirmation:")
print("  All sections in the knowledge base are available for extraction.")
print("  The extractor scans ALL categories during skill extraction.")
print(f"\n  Total sections scanned: {len(kb_sections)}")
print(f"  Total unique skills available: {len(extractor.kb):,}")
print("="*70)

## Export Results (Optional)

Save your test results to a file:

In [ ]:
# Export to JSON
output_file = '../data/test_extraction_results.json'

export_data = {
    'input_text': my_text.strip(),
    'parameters': {
        'similarity_threshold': SIMILARITY_THRESHOLD,
        'max_window_size': MAX_WINDOW_SIZE,
        'model': 'all-MiniLM-L6-v2'
    },
    'results': {
        'total_skills': my_result['num_skills'],
        'skills': sorted(my_result['skills']),
        'by_section': {k: sorted(v) for k, v in my_result['by_section'].items()},
        'details': my_result.get('details', [])
    }
}

with open(output_file, 'w', encoding='utf-8') as f:
    json.dump(export_data, f, indent=2, ensure_ascii=False)

print(f"✓ Results exported to: {output_file}")

## Summary

This notebook allows you to:
- ✅ Manually input any text for testing
- ✅ See all KSAO categories that are scanned
- ✅ View detailed extraction results with similarity scores
- ✅ Adjust parameters (similarity threshold)
- ✅ Compare multiple texts
- ✅ Verify all categories in knowledge base are being used

**Next steps:**
1. Test with your own job descriptions
2. Adjust similarity threshold based on results
3. Use `notebooks/extract_skills_from_jd.ipynb` for batch processing